### Hydrothermal Processing ML Pipeline with Uncertainty Quantification



In [59]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
src_dir = project_root / "src"
sys.path.insert(0, str(src_dir))

DATA_DIR = project_root / "data"
MODELS_DIR = project_root / "models"
OUTPUTS_DIR = project_root / "outputs"
CONFORMAL_DIR = OUTPUTS_DIR / "conformal"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

CONFORMAL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Data: {DATA_DIR}")
print(f"Models: {MODELS_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")


Project root: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline
Data: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/data
Models: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/models
Outputs: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/outputs


#### Data loading and initial exploration
We start by loading the HTT dataset and inspecting its structure, size, and basic statistics to understand the available information.



In [60]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import triang, t
from pathlib import Path
import seaborn as sns
import os


In [61]:

csv_path = (Path.cwd() / "../data/HTT_normalized_data_catalysts.csv").resolve()
htt_data = pd.read_csv(csv_path)

print(len(htt_data))


3693


In [62]:

subset_families = [
    "Agricultural Residues",
    "Herbaceous Biomass",
    "Woody Biomass / Hardwood",
    "Woody Biomass / Softwood",
    "Woody Biomass / Unspecified",
    "Lignin-rich Streams (Technical)",
]

#htt_data = htt_data[htt_data["Family"].isin(subset_families)].copy()

print("Selected rows:", len(htt_data))
print(htt_data["Family"].value_counts())


Selected rows: 3693
Family
Agricultural Residues              1376
Woody Biomass / Softwood            524
Woody Biomass / Hardwood            501
Mixed Biomass                       342
Lignin-rich Streams (Technical)     317
Unknown                             163
Animal Manure                       124
Waste Biomass / Sludge               92
Woody Biomass / Unspecified          71
Herbaceous Biomass                   60
Aquatic Biomass                      45
Waste Wood / Construction            36
Plastics / Polymer Blends            32
Model Compound                       10
Name: count, dtype: int64


In [63]:
#"""
valid_subtypes = ["HTL", "HTL_solvothermal"] #,, "HTC"
#htt_data = htt_data[htt_data["process_subtype"].isin(valid_subtypes)]
print("Selected rows:", len(htt_data))
print(htt_data["process_subtype"].value_counts())
#"""

Selected rows: 3693
process_subtype
HTL                 2166
HTL_solvothermal     994
HTC                  524
APR                    9
Name: count, dtype: int64


In [64]:
USE_INTERACTIONS = True

if "interaction_features" in sys.modules:
    del sys.modules["interaction_features"]

from interaction_features import add_interaction_features, INTERACTION_COLS

if USE_INTERACTIONS:
    INTERACTION_COLS = add_interaction_features(htt_data, use_elemental=True)
    print(f"Added {len(INTERACTION_COLS)} interaction features")
else:
    INTERACTION_COLS = []


Added 34 interaction features


In [65]:
import os
import pandas as pd
import importlib
from normalize_daf_to_dry_basis import normalize_daf_to_dry_basis
importlib.reload(importlib.import_module("normalize_daf_to_dry_basis"))

#htt_data = normalize_daf_to_dry_basis(htt_data)

<module 'normalize_daf_to_dry_basis' from '/home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/src/normalize_daf_to_dry_basis.py'>

In [66]:
USE_TIER   = True
USE_FAMILY = True
USE_BASIS  = False
USE_INTERACTIONS = True

DEBUG_FEATURE_CFG = False

BASE_X_NUM = [
    "O/C", "H/C", "Ash", "HHV_feedstock", "O", "C", "H",
    "T", "t", "IC",
    "Lignin", "cellulose", "hemicellulose", "Holocellulose",
    "LRI_imputed",# "Lignin_est", "Lignin_HHV_est", "Lignin_C_est",
    #"LRI_dd", "CeRI_dd", "HeRI_dd",
    "pressure_effective_mpa", "catalyst_biomass_ratio",
]

CAND_X_CAT = [
    "Cat_alkaline", "Cat_metal", "Cat_acid", "Cat_oxide", "Cat_zeolite",
    "Cat_none", "Cat_other", "Cat_salt", "Cat_sulfide", "Cat_support",
    "Cat_additive", "Cat_phosphide",
    "Cat_is_base", "Cat_base_strong", "Cat_base_carbonate",
    "Cat_is_acid", "Cat_acid_strong", "Cat_acid_weak",
    "Cat_is_redox", "Cat_has_noble", "Cat_has_base_metal",
    "Cat_bimetallic", "Cat_is_zeolite", "Cat_is_support",
    "Cat_is_hdonor",
    "cat_ratio_log10",
]

if "INTERACTION_COLS" not in globals():
    INTERACTION_COLS = []
    USE_INTERACTIONS = False

CAND_X_NUM = BASE_X_NUM + CAND_X_CAT + (INTERACTION_COLS if USE_INTERACTIONS else [])

CAND_X_SOLV_NUM = [
    "frac_water", "frac_ethanol", "frac_methanol",
    "frac_isopropanol", "frac_acetone", "frac_glycerol",
    "acid_M", "base_M", "phenol_additive_wt_pct",
]
CAND_X_SOLV_FLAG = ["acid_flag", "base_flag", "H_donor_flag"]

COLS_Y = [
    "B_Y", "C_Y", "A_Y", "G_Y",
    "E_B", "E_H", "C_B", "C_H",
    "HHV_biooil",
    "C_biooil", "O_biooil", "H_biooil", "N_biooil", "S_biooil",
    "HHV_biochar",
    "C_biochar", "O_biochar", "H_biochar", "N_biochar", "S_biochar",
    "O/C_biooil", "H/C_biooil",
    "O/C_biochar", "H/C_biochar",
]

print(f"Config: TIER={USE_TIER} | FAMILY={USE_FAMILY} | BASIS={USE_BASIS} | INTER={USE_INTERACTIONS}")
print(f"Features: base={len(BASE_X_NUM)} | cat={len(CAND_X_CAT)} | inter={len(INTERACTION_COLS)} | total={len(CAND_X_NUM)}")
print(f"Targets: {len(COLS_Y)}")


Config: TIER=True | FAMILY=True | BASIS=False | INTER=True
Features: base=17 | cat=26 | inter=34 | total=77
Targets: 24


In [67]:
from data_preparation import export_raw_data

export_raw_data(htt_data, export_dir="Testing")


EXPORTING RAW DATASET BEFORE DROP (WITH METADATA)

✅ Metadata exported:
   File: Testing/metadata_full_before_drop.csv
   Rows: 3693
   Columns: ['DOI', 'year', 'Feedstock', 'Family', 'Family_std', 'Tier', 'T', 't', 'process_type', 'Process_type', 'process_subtype', 'process_subtype_raw', 'paper_title', 'Ref']

✅ Numeric features + metadata exported:
   File: Testing/numeric_features_full_before_drop.csv
   Rows: 3693
   Columns: 27
   📋 Metadata columns: ['DOI', 'paper_title', 'Feedstock', 'Ref']
   📊 LRI_imputed non-zero values: 3693

✅ Targets exported: 3693 rows × 25 cols

✅ Categorical+encoding features exported: 3693 rows × 69 cols

RAW DATA EXPORT COMPLETE: 3693 rows



In [68]:
from data_preparation import build_training_features

X_for_testing, Y_for_testing, _ = build_training_features(
    htt_data,
    CAND_X_NUM=CAND_X_NUM,
    CAND_X_SOLV_NUM=CAND_X_SOLV_NUM,
    CAND_X_SOLV_FLAG=CAND_X_SOLV_FLAG,
    COLS_Y=COLS_Y,
    USE_TIER=USE_TIER,
    export_dir="Testing"
)

print(f"X_for_testing: {X_for_testing.shape}")
print(f"Y_for_testing: {Y_for_testing.shape}")
print(f"Files exported to ../outputs/tables/testing_")



BUILDING TRAINING FEATURES
✓ Found 24 solvent one-hot columns:
   ['Solv_acetone', 'Solv_alcohol', 'Solv_alcohol_ketone_mix', 'Solv_aqueous_organic', 'Solv_black_liquor', 'Solv_butanediol', 'Solv_butanediol+triethanolamine', 'Solv_carrier_oil', 'Solv_ethanol', 'Solv_ethanol_sc', 'Solv_ethylene_glycol', 'Solv_glycerol', 'Solv_isopropanol', 'Solv_methanol', 'Solv_methanol_sc', 'Solv_none', 'Solv_phenol', 'Solv_pyroligneous_acid', 'Solv_sc_co2', 'Solv_tetralin', 'Solv_toluene', 'Solv_triethanolamine', 'Solv_wastewater', 'Solv_water']
✓ Found 5 Basis_* columns for yield-basis encoding.
✓ Rows with complete core numeric features: 3385 / 3693
✓ Including 5 yield-basis one-hot features (e.g. ['Basis_carbon_basis', 'Basis_daf', 'Basis_dry_basis']...)
✓ Including 12 catalyst features (coarse classes + descriptor flags)
✓ Including 34 interaction features
⚠️ Removed 46 duplicate columns
✓ Engineered X shape: (3385, 136)
✓ Target Y shape: (3385, 24) (targets only)
✓ Y with grouping columns: (338

In [69]:
def normalize_doi(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    s = s.lower()
    prefixes = [
        "https://doi.org/",
        "http://doi.org/",
        "http://dx.doi.org/",
        "dx.doi.org/",
        "doi:",
    ]
    for p in prefixes:
        if s.startswith(p):
            s = s[len(p):]
            break
    return s
if "DOI" in htt_data.columns:
    htt_data["DOI"] = htt_data["DOI"].apply(normalize_doi)

In [70]:

TEST_PAPERS = {
    "Duan_2022": {
        "label": "Duan 2022 – COA + Spirulina HTL",
        "doi": "10.1016/j.renene.2021.12.071",
        "title": (
            "Synergistic effect of hydrothermal co-liquefaction of Camellia oleifera Abel and "
            "Spirulina platensis: Parameters optimization and product characteristics"
        ),
    },
    "Zhang_2019": {
        "label": "Zhang 2019 – corn stover HTC",
        "doi": "10.1016/j.biombioe.2019.01.035",
        "title": (
            "Effects of temperature, time and acidity of hydrothermal carbonization on the "
            "hydrochar properties and nitrogen recovery from corn stover"
        ),
    },
    "Zhao_2020": {
        "label": "Zhao 2020 – lignite + biomass + formic acid",
        "doi": "10.1021/acs.iecr.0c04619",
        "title": (
            "Hydrothermal Co-Liquefaction of Lignite and Lignocellulosic Biomass with the "
            "Addition of Formic Acid: Study on Product Distribution, Characteristics, "
            "and Synergistic Effects"
        ),
    },
    "kaur_2020": {
        "label": "Kaur residue HTL 2020",
        "doi": "10.1016/j.indcrop.2020.112359",
        "title": "Catalytic hydrothermal liquefaction of castor residue to bio-oil: "
                 "Effect of alkali catalysts and optimization study",
    },
    "Hardi_2017": {
        "label": "Hardi 2017 – pine sawdust HTL",
        "doi": "10.1016/j.apenergy.2017.04.033",
        "title": "Non-catalytic hydrothermal liquefaction of pine sawdust using "
                 "experimental design: Material balances and products analysis",
    },
    "Huang_2013": {
        "label": "Huang 2013 – rice husk HTL",
        "doi": "10.1016/j.jaap.2013.04.002",
        "title": "Thermochemical liquefaction of rice husk for bio-oil production "
            "with sub- and supercritical ethanol as solvent",
    },
    "Wang_2023": {
        "label": "Wang 2023 – Global kinetic model for HTL yields",
        "doi": "10.1016/j.renene.2023.118956",
        "title": "Development of a global kinetic model based on chemical compositions of lignocellulosic biomass for predicting product yields from hydrothermal liquefaction",
    },
    "He_2020": {
        "label": "He 2020 – HTL of SP nad Lignin",
        "doi": "10.1016/j.energy.2020.117550",
        "title": "Synergistic effect of hydrothermal Co-liquefaction of Spirulina platensis and Lignin: Optimization of operating parameters by response surface methodology",
}
}

### Full paper drop

In [71]:
import pandas as pd

# ========================================
# DROP TEST DATA (EXCLUDE FROM TRAINING)
# ========================================
# Test data:
#    remains in raw exports (Cell 11) for later benchmarking
#   ✗ is excluded from training (htt_data)
# ========================================

drop_for_testing = False




# --------------------------------------------------------------------------
# 2) Choose which papers to hold out as test data
#    (edit this list in the notebook as needed)
# --------------------------------------------------------------------------
SELECT_TEST_PAPERS = [
    #"Hardi_2017"
    #"Zhang_2019",
    #"Duan_2022",
    #"Zhao_2020",
    "Huang_2013",
    #"kaur_2020",
    #"Wang_2023"
    #"He_2020",
]


EXTRA_TEST_ENTRIES = [
    # {"doi": "10.xxx/yyy", "title": "Exact full title here", "label": "My custom test paper"},
]


if drop_for_testing:
    paper_title_col = "Paper_Title" if "Paper_Title" in htt_data.columns else "paper_title"

   
    doi_series = htt_data["DOI"].astype(str).str.strip()
    title_series = htt_data[paper_title_col].astype(str).str.strip()

    rows_before = len(htt_data)
    drop_mask = pd.Series(False, index=htt_data.index)

    dropped_labels = []

 
    for key in SELECT_TEST_PAPERS:
        if key not in TEST_PAPERS:
            print(f" Test key '{key}' not found in TEST_PAPERS – skipping.")
            continue

        meta = TEST_PAPERS[key]
        doi = str(meta.get("doi", "")).strip()
        title = str(meta.get("title", "")).strip()
        label = meta.get("label", key)

        m = pd.Series(False, index=htt_data.index)
        if doi:
            m |= doi_series == doi
        if title:
            m |= title_series == title

        if m.any():
            drop_mask |= m
            dropped_labels.append(label)
        else:
            print(f" No rows matched for '{label}' (doi='{doi}')")

 
    for meta in EXTRA_TEST_ENTRIES:
        doi = str(meta.get("doi", "")).strip()
        title = str(meta.get("title", "")).strip()
        label = meta.get("label", doi or title or "custom")

        m = pd.Series(False, index=htt_data.index)
        if doi:
            m |= doi_series == doi
        if title:
            m |= title_series == title

        if m.any():
            drop_mask |= m
            dropped_labels.append(label)
        else:
            print(f" No rows matched for custom entry '{label}'")


    htt_data = htt_data[~drop_mask]
    rows_after = len(htt_data)
    rows_dropped = rows_before - rows_after

    print(f"Rows before drop: {rows_before}")
    print(f"Rows after drop:  {rows_after}")
    print(f"Rows dropped:     {rows_dropped}")

    if dropped_labels:
        print("\n Test data will be EXCLUDED from training for:")
        for lab in dropped_labels:
            print(f"  - {lab}")
        print("\n These rows remain present in the raw exports (Cell 11) for later testing.")
    else:
        print("\n No test rows were dropped (check DOIs/titles or TEST_PAPERS).")


#### Partial drop (acceleration of experimental compaigns / Pre-trained model tuning)

In [72]:
partial_drop_for_testing = False

HE2020_DOI = "10.1016/j.energy.2020.117550"
HE2020_TABLE8_COND = {(320.0, 20.0), (330.0, 30.0), (340.0, 40.0), (350.0, 50.0), (360.0, 60.0)}
HE2020_TABLE11_COND = {(320.0, 20.0), (340.0, 40.0), (360.0, 60.0)}

HE2020_TABLE8_FEEDSTOCKS = [
    "Blend: Spirulina/Lignin (B.R=0; pure lignin)",
    "Blend: Spirulina/Lignin (B.R=0.5)",
    "Blend: Spirulina/Lignin (B.R=1)",
    "Blend: Spirulina/Lignin (B.R=1.5)",
    "Blend: Spirulina/Lignin (B.R=2)",
    "Blend: Spirulina/Lignin (B.R=+inf; pure Spirulina)",
]

HE2020_TABLE11_FEEDSTOCKS = [
    "Blend: Spirulina/Lignin (B.R=0; pure lignin)",
    "Blend: Spirulina/Lignin (B.R=2)",
    "Blend: Spirulina/Lignin (B.R=+inf; pure Spirulina)",
]

def _as_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def _pair_Tt(df):
    T = df["T"].map(_as_float)
    t = df["t"].map(_as_float)
    return pd.Series(list(zip(T, t)), index=df.index)

def _mask_he2020_table8(df):
    doi_m = df["DOI"].astype(str).str.strip().eq(HE2020_DOI)
    fs_m = df["Feedstock"].astype(str).str.strip().isin(HE2020_TABLE8_FEEDSTOCKS)
    Tt_m = _pair_Tt(df).isin(HE2020_TABLE8_COND)
    return doi_m & fs_m & Tt_m

def _mask_he2020_table11(df):
    doi_m = df["DOI"].astype(str).str.strip().eq(HE2020_DOI)
    fs_m = df["Feedstock"].astype(str).str.strip().isin(HE2020_TABLE11_FEEDSTOCKS)
    Tt_m = _pair_Tt(df).isin(HE2020_TABLE11_COND)
    hhv_m = df["HHV_biooil"].notna()
    return doi_m & fs_m & Tt_m & hhv_m

SELECT_PARTIAL_TESTS = ["He_2020_Table8_Table11"]

if partial_drop_for_testing:
    rows_before = len(htt_data)
    
    if "_partial_test_tag" not in htt_data.columns:
        htt_data["_partial_test_tag"] = None
    else:
        htt_data["_partial_test_tag"] = None
    
    drop_mask = pd.Series(False, index=htt_data.index)
    
    for key in SELECT_PARTIAL_TESTS:
        if key == "He_2020_Table8_Table11":
            m8 = _mask_he2020_table8(htt_data)
            m11 = _mask_he2020_table11(htt_data)
            
            htt_data.loc[m8, "_partial_test_tag"] = "He2020/Table8"
            htt_data.loc[m11, "_partial_test_tag"] = "He2020/Table11_subset"
            drop_mask |= (m8 | m11)
            
            print(f"He2020 Table8 matched: {int(m8.sum())} rows")
            print(f"He2020 Table11 matched: {int(m11.sum())} rows")
    
    htt_data = htt_data.loc[~drop_mask].copy()
    rows_after = len(htt_data)
    print(f"Rows before: {rows_before}")
    print(f"Rows after: {rows_after}")
    print(f"Rows dropped: {rows_before - rows_after}")


In [73]:
col_map = {
    "O/C": "O/C",
    "H/C": "H/C",
    "Ash": "Ash",
    "Temperature": "T",
    "Time": "t",
    "Solid": "IC",
    "oil_yield": "B_Y",
    "char_yield": "C_Y",
    "aqueous_yield": "A_Y",
    "gas_yield": "G_Y",
    "oil_HHV": "E_B",
    "char_HHV": "E_H",
    "oil_C": "C_B",
    "char_C": "C_H"
    }
htt_data =htt_data.rename(columns=col_map)


In [74]:
import numpy as np
import pandas as pd
import sys

if "data_preparation" in sys.modules:
    del sys.modules["data_preparation"]

from data_preparation import build_training_features

X, Y, df_Xok = build_training_features(
    htt_data,
    CAND_X_NUM=CAND_X_NUM,
    CAND_X_SOLV_NUM=CAND_X_SOLV_NUM,
    CAND_X_SOLV_FLAG=CAND_X_SOLV_FLAG,
    COLS_Y=COLS_Y,
    USE_TIER=USE_TIER,
    USE_FAMILY=USE_FAMILY,
    USE_BASIS=USE_BASIS,
    export_dir="Training",
)

if "rf_trainers" in sys.modules:
    del sys.modules["rf_trainers"]

from rf_trainers import (
    train_auto_from_csv_default,
    train_auto_groupkfold,
    train_auto_groupkfold_picklesafe,
    tune_rf_defaults_oob,
    RF_DEFAULT,
    _rows_for_target,
)

X_t, y_t = _rows_for_target(X, Y, "B_Y")
best_rf = tune_rf_defaults_oob(X_t, y_t, base_defaults=RF_DEFAULT)

print(f"Complete-X samples: {len(df_Xok)} / {len(htt_data)}")
print(f"X shape: {X.shape} | Y shape: {Y.shape}")

def _safe_binary_count(df: pd.DataFrame, col_name: str) -> tuple[int, float]:
    sub = df.loc[:, col_name]
    if isinstance(sub, pd.DataFrame):
        arr = sub.to_numpy()
        n_ones = float(np.nansum(arr))
    else:
        n_ones = float(sub.sum())
    n_rows = len(df)
    pct = (n_ones / n_rows * 100.0) if n_rows > 0 else 0.0
    return int(round(n_ones)), pct

cat_cols_in_X = [c for c in X.columns if c.startswith("Cat_")]
unique_cat_cols = sorted(set(cat_cols_in_X))
print(f"Catalyst features: {len(unique_cat_cols)} columns")

int_cols_in_X = [c for c in X.columns if c.startswith("INT_")]
print(f"Interaction features: {len(int_cols_in_X)} columns")

if "Family" in htt_data.columns:
    family_in_X = [c for c in X.columns if c.startswith("Family_")]
    family_std_in_X = [c for c in X.columns if c.startswith("FamilyStd_")]
    print(f"Family features: {len(family_in_X)} raw, {len(family_std_in_X)} std")

basis_in_X = [c for c in X.columns if c.startswith("Basis_")]
print(f"Basis features: {len(set(basis_in_X))} columns")

ml_ready = {ycol: int((~Y[ycol].isna()).sum()) for ycol in COLS_Y if ycol in Y.columns}
print(f"ML-ready targets: {len(ml_ready)}")



BUILDING TRAINING FEATURES
✓ Found 24 solvent one-hot columns:
   ['Solv_acetone', 'Solv_alcohol', 'Solv_alcohol_ketone_mix', 'Solv_aqueous_organic', 'Solv_black_liquor', 'Solv_butanediol', 'Solv_butanediol+triethanolamine', 'Solv_carrier_oil', 'Solv_ethanol', 'Solv_ethanol_sc', 'Solv_ethylene_glycol', 'Solv_glycerol', 'Solv_isopropanol', 'Solv_methanol', 'Solv_methanol_sc', 'Solv_none', 'Solv_phenol', 'Solv_pyroligneous_acid', 'Solv_sc_co2', 'Solv_tetralin', 'Solv_toluene', 'Solv_triethanolamine', 'Solv_wastewater', 'Solv_water']
✓ Rows with complete core numeric features: 3385 / 3693
✓ Including 12 catalyst features (coarse classes + descriptor flags)
✓ Including 34 interaction features
⚠️ Removed 46 duplicate columns
✓ Engineered X shape: (3385, 131)
✓ Target Y shape: (3385, 24) (targets only)
✓ Y with grouping columns: (3385, 28) (targets + ['DOI', 'Ref', 'Feedstock', 'T'] for grouping/filtering)

EXPORTING ENGINEERED FEATURES (AFTER FEATURE BUILDING)

✅ Engineered features + meta

In [75]:
def decode_family_from_ohe(X: pd.DataFrame, prefix="Family_"):
    fam_cols = [c for c in X.columns if c.startswith(prefix)]
    
    if not fam_cols:
        raise ValueError(f"No family columns found with prefix '{prefix}'")
    
    fam_values = X[fam_cols].idxmax(axis=1).str.replace(prefix, "", regex=False)
    
    all_zero = (X[fam_cols].sum(axis=1) == 0)
    fam_values[all_zero] = "Unknown"
    
    fam_values = fam_values.replace(["nan", ""], "Unknown")
    
    return fam_values


In [76]:
print(X.columns.to_list())

['O/C', 'H/C', 'Ash', 'HHV_feedstock', 'O', 'C', 'H', 'T', 't', 'IC', 'Lignin', 'cellulose', 'hemicellulose', 'Holocellulose', 'LRI_imputed', 'pressure_effective_mpa', 'catalyst_biomass_ratio', 'Cat_alkaline', 'Cat_metal', 'Cat_acid', 'Cat_oxide', 'Cat_zeolite', 'Cat_none', 'Cat_other', 'Cat_salt', 'Cat_sulfide', 'Cat_support', 'Cat_additive', 'Cat_phosphide', 'INT_T_IC', 'INT_T_IC_missing', 'INT_T_pressure', 'INT_T_pressure_missing', 'INT_IC_cat_ratio', 'INT_IC_cat_ratio_missing', 'INT_T_Lignin', 'INT_T_Lignin_missing', 'INT_T_cellulose', 'INT_T_cellulose_missing', 'INT_T_hemicellulose', 'INT_T_hemicellulose_missing', 'INT_T_H_over_C', 'INT_T_H_over_C_missing', 'INT_T_O_over_C', 'INT_T_O_over_C_missing', 'INT_T_LRI', 'INT_T_LRI_missing', 'INT_T_cat_ratio', 'INT_T_cat_ratio_missing', 'INT_T_C', 'INT_T_C_missing', 'INT_T_H', 'INT_T_H_missing', 'INT_T_O', 'INT_T_O_missing', 'INT_T_Ash', 'INT_T_Ash_missing', 'INT_Ash_Lignin', 'INT_Ash_Lignin_missing', 'INT_Ash_LRI', 'INT_Ash_LRI_missing',

### Model Training Overview

This pipeline trains Random Forest models to predict HTT product yields and properties.

#### Input Features

Core features (from feedstock and process conditions):

| Variable | Description |
|----------|-------------|
| `O/C`, `H/C` | Elemental ratios |
| `Ash` | Ash content (wt%) |
| `T`, `t` | Temperature (°C), time (min) |
| `IC` | Initial solid concentration (%) |

Additional features include catalyst properties, solvent fractions, and interaction terms.

#### Output Targets

Models predict 24 targets including:
- Product yields: `B_Y`, `C_Y`, `A_Y`, `G_Y`
- Energy recovery: `E_B`, `E_H`
- Carbon distribution: `C_B`, `C_H`
- Elemental composition: `C_biooil`, `H_biooil`, `O_biooil`, etc.
- Atomic ratios: `O/C_biooil`, `H/C_biooil`, etc.

#### Training Strategy

- Model: RandomForest (sklearn)
- Validation: GroupKFold cross-validation (grouped by DOI to avoid data leakage)
- Hyperparameters: Grid search or pre-tuned parameters from CSV
- Metrics: R², RMSE (cross-validated)

Each target is trained independently to handle missing data patterns.


### Training Configuration

#### Hyperparameter Strategy

**Option 1: Use saved best parameters (recommended for production)**
```python
USE_BEST_PARAMS = True
DO_GRID_SEARCH = False
BEST_PARAMS_CSV = "../outputs/tables/best_params_global_with_tier_subprocess_catalysts_lri_gs.csv"
```
- Loads pre-tuned hyperparameters from CSV
- Fast training (no search overhead)
- Requires CSV file to exist

**Option 2: Perform grid search (for new datasets or re-tuning)**
```python
USE_BEST_PARAMS = False
DO_GRID_SEARCH = True
```
- Runs hyperparameter grid search for each target
- Saves results to BEST_PARAMS_CSV
- Slow but finds optimal parameters

#### Trainer Mode

**`TRAINER_MODE` options:**

1. `"default"`: Standard cross-validation (random splits)
2. `"group"`: GroupKFold cross-validation (groups by DOI)
3. `"picklesafe"`: GroupKFold + saves conformal artifacts (required for UQ)

**Recommendation**: Use `TRAINER_MODE="picklesafe"` to enable all UQ methods.

#### Model Selection

**`MODEL_MODE` options:**

| Mode | Description | Ensemble UQ | Conformal UQ |
|------|-------------|-------------|---------------|
| `"rf_qr"` | RandomForest quantile regression | Yes | Yes |
| `"rf"` | Standard RandomForest | Yes | Yes |
| `"rf_cal"` | RF with conformal calibration | Yes | Yes |
| `"extratrees"` | ExtraTrees ensemble | Yes | Yes |
| `"xgb"` | XGBoost | No | Yes |
| `"ngb"` | NGBoost | No | Yes |

**Recommendation**: Use `MODEL_MODE="rf_qr"` for full UQ support.

### Uncertainty Quantification

#### Available UQ Methods

**1. Ensemble UQ (`uq_mode="ensemble"`)**
- Uses tree-level predictions from RandomForest
- Fast, requires forest-based model (RF/ExtraTrees)
- Good for interpolation, may underestimate in extrapolation

**2. Global Conformal (`uq_mode="conformal"`)**
- Single quantile threshold from calibration residuals
- Distribution-free coverage guarantee
- Uniform width (conservative in well-modeled regions)
- Files: `../outputs/conformal/rf_conf_<target>.json`, `*_abs_resid.npy`

**3. Local Conformal (`uq_mode="local_conformal"`)**
- Adaptive intervals using k-NN in feature space
- Wider in extrapolation/sparse regions
- Best balance of coverage and precision
- Files: `../outputs/conformal/rf_conf_local_<target>.npz`

#### Practical Usage

```python
# Select UQ method
uq_mode = "local_conformal"  # or "ensemble" or "conformal"

diag = deep_uq_diagnostics(
    target="B_Y",
    X_test=X_test,
    Y_test=Y_test,
    alpha=0.10,  # 90% PI
    uq_mode=uq_mode,
    conformal_dir="../outputs/conformal",
    make_plots=True,
)
```

#### Decision Guide

- **Interpolation (within training range)**: Use `"ensemble"` for speed
- **Extrapolation or new conditions**: Use `"local_conformal"` for adaptive intervals
- **Guaranteed coverage**: Use `"conformal"` for statistical guarantee
- **XGBoost model**: Only conformal methods available

All conformal files are auto-generated when `TRAINER_MODE="picklesafe"`.


In [77]:
import os
import sys
import numpy as np
import pandas as pd

if "rf_trainers" in sys.modules:
    del sys.modules["rf_trainers"]

from rf_trainers import (
    train_auto_from_csv_default,
    train_auto_groupkfold,
    train_auto_groupkfold_picklesafe,
)

if "COLS_Y" not in globals():
    raise RuntimeError("COLS_Y not defined. Re-run Feature Build cell.")

EXPECTED_N_TARGETS = 24
if len(COLS_Y) != EXPECTED_N_TARGETS:
    raise RuntimeError(f"COLS_Y has {len(COLS_Y)} targets, expected {EXPECTED_N_TARGETS}")

print(f"Training {len(COLS_Y)} targets")

DO_SPLIT = True
SPLIT_MODE = "random_rows"
TEST_FRACTION = 0.2
SPLIT_COL = "DOI"
SPLIT_CSV = "../outputs/tables/train_test_split_by_DOI.csv"
FIX_SPLIT = False

if DO_SPLIT:
    rs = np.random.RandomState(42)
    
    if SPLIT_MODE == "doi":
        split_path = SPLIT_CSV
        
        if FIX_SPLIT and os.path.exists(split_path):
            split_df = pd.read_csv(split_path, index_col="row_index")
            if not split_df.index.equals(Y.index):
                raise RuntimeError("Split index mismatch")
            train_mask = split_df["is_train"].astype(bool)
            test_mask = split_df["is_test"].astype(bool)
        else:
            if SPLIT_COL not in Y.columns:
                raise RuntimeError(f"Column {SPLIT_COL} not found in Y")
            
            unique_doi = pd.Series(Y[SPLIT_COL].dropna().unique()).astype(str)
            doi_shuffled = unique_doi.sample(frac=1.0, random_state=42).values
            n_test_doi = max(1, int(round(TEST_FRACTION * len(doi_shuffled))))
            test_dois = set(doi_shuffled[:n_test_doi])
            
            test_mask = Y[SPLIT_COL].astype(str).isin(test_dois)
            train_mask = ~test_mask
            
            split_df = pd.DataFrame(
                {"is_train": train_mask, "is_test": test_mask, SPLIT_COL: Y[SPLIT_COL].astype(str)},
                index=Y.index
            )
            os.makedirs(os.path.dirname(split_path), exist_ok=True)
            split_df.to_csv(split_path, index_label="row_index")
            print(f"Saved split to {split_path}")
    
    elif SPLIT_MODE == "random_rows":
        out_csv = SPLIT_CSV.replace("_by_DOI", "_random_rows")
        
        if FIX_SPLIT and os.path.exists(out_csv):
            split_df = pd.read_csv(out_csv, index_col="row_index")
            train_mask = split_df["is_train"].astype(bool)
            test_mask = split_df["is_test"].astype(bool)
        else:
            mask_rand = rs.rand(len(Y)) >= TEST_FRACTION
            train_mask = pd.Series(mask_rand, index=Y.index)
            test_mask = ~train_mask
            
            split_df = pd.DataFrame(
                {"is_train": train_mask, "is_test": test_mask,
                 SPLIT_COL: Y.get(SPLIT_COL, pd.Series(index=Y.index, dtype=str))},
                index=Y.index
            )
            os.makedirs(os.path.dirname(out_csv), exist_ok=True)
            split_df.to_csv(out_csv, index_label="row_index")
            print(f"Saved split to {out_csv}")
    
    X_train = X.loc[train_mask].copy()
    Y_train = Y.loc[train_mask].copy()
    X_test = X.loc[test_mask].copy()
    Y_test = Y.loc[test_mask].copy()
    
    print(f"Train: {len(X_train)} | Test: {len(X_test)}")
else:
    X_train = X.copy()
    Y_train = Y.copy()
    X_test = pd.DataFrame(index=[], columns=X.columns)
    Y_test = pd.DataFrame(index=[], columns=Y.columns)

USE_BEST_PARAMS = True
DO_GRID_SEARCH = False
BEST_PARAMS_CSV = str(TABLES_DIR / "best_params_global_with_tier_subprocess_catalysts_lri_gs.csv")

groups_full = Y_train.get("DOI", pd.Series(index=Y_train.index, dtype=str)).copy()
groups_full = groups_full.fillna("UNKNOWN_DOI")

TRAINER_MODE = "default"   # "default", "group", "picklesafe"
MODEL_MODE   = "rf_qr"        # "rf_qr", "rf", "extratrees", "xgb", "ngb"
SAVE_DIR = str(MODELS_DIR / "saved_models_global_with_tier_subprocess_catalyst_lri")
CONFORMAL_DIR = str(OUTPUTS_DIR / "conformal")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CONFORMAL_DIR, exist_ok=True)

print(f"Trainer: {TRAINER_MODE} | Model: {MODEL_MODE}")
print(f"Models: {SAVE_DIR}")
print(f"Conformal: {CONFORMAL_DIR}")

if TRAINER_MODE == "default":
    results_df_cv = train_auto_from_csv_default(
        X_train, Y_train,
        save_dir=SAVE_DIR,
        best_params_csv=BEST_PARAMS_CSV,
        cols_y=COLS_Y,
        use_best_params=USE_BEST_PARAMS,
        model_mode=MODEL_MODE,
        do_grid_search=DO_GRID_SEARCH,
    )
elif TRAINER_MODE == "group":
    results_df_cv = train_auto_groupkfold(
        X_train, Y_train,
        groups=groups_full,
        save_dir=SAVE_DIR,
        best_params_csv=BEST_PARAMS_CSV,
        cols_y=COLS_Y,
        use_best_params=USE_BEST_PARAMS,
        model_mode=MODEL_MODE,
        do_grid_search=DO_GRID_SEARCH,
    )
elif TRAINER_MODE == "picklesafe":
    results_df_cv = train_auto_groupkfold_picklesafe(
        X_train, Y_train,
        groups=groups_full,
        save_dir=SAVE_DIR,
        best_params_csv=BEST_PARAMS_CSV,
        conformal_dir=CONFORMAL_DIR,
        cols_y=COLS_Y,
        use_best_params=USE_BEST_PARAMS,
        model_mode=MODEL_MODE,
    )

print("Cross-validated performance (train subset):")
print(results_df_cv.to_string(index=False))

if len(results_df_cv) == EXPECTED_N_TARGETS:
    print(f"All {EXPECTED_N_TARGETS} targets trained")
else:
    missing = [t for t in COLS_Y if t not in results_df_cv["Target"].values]
    print(f"Warning: Missing targets: {missing}")


Training 24 targets
Saved split to ../outputs/tables/train_test_split_random_rows.csv
Train: 2688 | Test: 697
Trainer: default | Model: rf_qr
Models: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/models/saved_models_global_with_tier_subprocess_catalyst_lri
Conformal: /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/outputs/conformal
ℹ️ No existing best_params CSV found; will start from RF_DEFAULT.
✓ RF grid search enabled; best params will be written back to CSV.
✓ model_mode = rf_qr


GS B_Y: best R²≈0.823
Training B_Y (n=2173, folds=10)... R²=0.823
GS C_Y: best R²≈0.857
Training C_Y (n=1954, folds=10)... R²=0.857
GS A_Y: best R²≈0.825
Training A_Y (n=679, folds=10)... R²=0.825
GS G_Y: best R²≈0.870
Training G_Y (n=811, folds=10)... R²=0.870
GS E_B: best R²≈0.739
Training E_B (n=864, folds=10)... R²=0.739
GS E_H: best R²≈0.901
Training E_H (n=629, folds=10)... R²=0.901
GS C_B: best R²≈0.777
Training C_B (n=865, folds=10)... R²=0.769
GS C_H: best R²≈0.886
Training C_H (n=543, folds=10)... R²=0.884
GS HHV_biooil: best R²≈0.704
Training HHV_biooil (n=971, folds=10)... R²=0.705
GS C_biooil: best R²≈0.684
Training C_biooil (n=951, folds=10)... R²=0.656
GS O_biooil: best R²≈0.705
Training O_biooil (n=892, folds=10)... R²=0.702
GS H_biooil: best R²≈0.647
Training H_biooil (n=917, folds=10)... R²=0.635
GS N_biooil: best R²≈0.823
Training N_biooil (n=664, folds=10)... R²=0.810
GS S_biooil: best R²≈0.801
Training S_biooil (n=647, folds=10)... R²=0.742
GS HHV_biochar: best R²≈

In [79]:
import os


results_output_path = os.path.join(TABLES_DIR, f"results_{TRAINER_MODE}.csv")


standard_name = "results_global_with_tier_subprocess_catalysts_lri.csv"
standard_path = os.path.join(TABLES_DIR, standard_name)


results_df_cv.to_csv(results_output_path, index=False)
results_df_cv.to_csv(standard_path, index=False)

print(f" Results saved to:")
print(f"   {results_output_path}")
print(f"   {standard_path}")
print(f"\n Summary:")
print(f"   Targets: {len(results_df_cv)}")
print(f"   Mean R² (RF): {results_df_cv['R2_RF'].mean():.3f}")
print(f"   Mean RMSE (RF): {results_df_cv['RMSE_RF'].mean():.3f}")

 Results saved to:
   /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/outputs/tables/results_default.csv
   /home/Elfetni/ML_UQ_LCA_pipeline/ml_uq_hydrothermal_pipeline/outputs/tables/results_global_with_tier_subprocess_catalysts_lri.csv

 Summary:
   Targets: 24
   Mean R² (RF): 0.772
   Mean RMSE (RF): 2.485
